In [9]:
import signac
import numpy as np
from Rogerson_et_al_2026_Virtual_Rishon_Formulation.simulations.simulation import DMRGSimulationRampChiConfig, DMRGSimulationRampChiAndDiagMethodConfig
from Rogerson_et_al_2026_Virtual_Rishon_Formulation.simulations.simulation import LatticeProductStateConfig
from Rogerson_et_al_2026_Virtual_Rishon_Formulation.models.massive_schwinger_model_qubit_encoding import MassiveSchwingerModelQubitEncodingConfig
from Rogerson_et_al_2026_Virtual_Rishon_Formulation.models.massive_schwinger_model_long_range import MassiveSchwingerModelLongRangeConfig
from Rogerson_et_al_2026_Virtual_Rishon_Formulation.models.QED3_qubit_encoding import QED3VRQubitEncodingConfig

project = signac.init_project("../")

project.detect_schema()

ProjectSchema(<len=0>)

# Initlize Massive Schwinger Model 

In [10]:
trunc_params = dict(
    chi_max=[16,32,64,128],
    degeneracy_tol=1e-10,
    trunc_cut=1e-10,
    svd_min=1e-14,
)
mixer_params = dict(
    amplitude=0.5,
    decay=5,
    disable_after=None,
)
dmrg_params=dict(
    trunc_params=trunc_params,
    mixer_params=mixer_params,
    max_sweeps=50,
    mixer=True,
    max_E_err=1e-9,
    max_S_err=1e-8,
)

import numpy as np
from itertools import product
Nf = 1
Nr = 4
Ls = [16]
bc_params = [('open', 'default', 'after_source'), ('periodic', 'folded', 'before_target')]
ms = [[0.3335]]


for L,m,(bc, order, gauge_order) in product(Ls, ms, bc_params):
    model_params = MassiveSchwingerModelQubitEncodingConfig(
        L=L,
        Nf=Nf,
        Nr=Nr,
        m=m,
        g=1.0,
        a=0.3,
        theta=np.pi,
        bc_MPS='finite',
        m_corr=True,
        bc_x=bc,
        order=order,
        gauge_order=gauge_order,
    )
    zero_rotor_state= [str(2**(i-1)) if i !=  Nr-1 else str(-2**(i-1)) for i in range(Nr-1, 0, -1)] + ['1'] if Nr != 1 else ['0']
    inital_state_params = LatticeProductStateConfig(product_state=    
        [['empty']*Nf + zero_rotor_state,['full']*Nf + zero_rotor_state]
    )
    config = DMRGSimulationRampChiConfig(
        model_class="MassiveSchwingerModelQubitEncoding",
        model_params=model_params.model_dump(),
        initial_state_params=inital_state_params,
        algorithm_class="TwoSiteDMRGEngine",
        algorithm_params=dmrg_params,
        save_psi=True,
    )
    # Create a signac project    
    # Create a signac job with the configuration
    job = project.open_job(statepoint=config.model_dump())
    job.doc['initilized'] = {'date': str(np.datetime64('now')), 'reason':"Minimal Example"}    # Write the configuration to the job's directory
    # Write the configuration to the job's directory
    job.init()


# Initilize Jobs for Long Range Schwinger Model

In [11]:
trunc_params = dict(
    chi_max=[16,32,64,128],
    degeneracy_tol=1e-10,
    trunc_cut=1e-10,
    svd_min=1e-14,
)
mixer_params = dict(
    amplitude=0.9,
    decay=5,
    disable_after=10,
)
dmrg_params=dict(
    trunc_params=trunc_params,
    mixer_params=mixer_params,
    max_sweeps=30,
    mixer=True,
    max_E_err=1e-9,
    max_S_err=1e-8,
    max_trunc_err=1,
)
Nf = 2
Ls = [16]

ms = [[0.0,0.0]]#[[m, m] for m in np.linspace(0, 1, 11)]
a_s = [0.3]
thetas = [np.pi, 0.]
n_jobs = 0
for a, L,m, theta in product(a_s, Ls, ms, thetas):
    n_jobs +=1
    model_params = MassiveSchwingerModelLongRangeConfig(
        L=L,
        Nf=Nf,
        m=m,
        g=1.0,
        a=0.3,
        bc_x='open',
        order='default',
        theta=theta,
        bc_MPS='finite',
        m_corr=True,
    )
    inital_state_params = LatticeProductStateConfig(product_state=    
        [['empty', 'empty'],['full', 'full']]
    )
    config = DMRGSimulationRampChiConfig(
        model_class="MassiveSchwingerModelLongRange",
        model_params=model_params.model_dump(),
        initial_state_params=inital_state_params,
        algorithm_class="TwoSiteDMRGEngine",
        algorithm_params=dmrg_params,
        save_psi=False,
    )
    # Create a signac project
    
    # Create a signac job with the configuration
    job = project.open_job(statepoint=config.model_dump())
    job.init()
    job.doc['initilized'] = {'date': str(np.datetime64('now')), 'reason':"Long Range Model Cross initial test"}    # Write the configuration to the job's directory
print("Created", n_jobs, "jobs") 

Created 2 jobs


# Initilize QED$_3$ String Tension Calculation 

In [12]:
trunc_params = dict(
    chi_max=[16,32, 64,128],
    degeneracy_tol=1e-10,
    trunc_cut=1e-10,
    svd_min=1e-14,
)
mixer_params = dict(
    amplitude=0.9,
    decay=2,
    disable_after=20,
)
dmrg_params=dict(
    trunc_params=trunc_params,
    mixer_params=mixer_params,
    max_sweeps=50,
    mixer=True,
    max_E_err=1e-9,
    max_S_err=1e-8,
    combine=True,
    max_trunc_err=1,
    max_N_sites_per_ring=None,
)

gs = [2.0]
ms = [None]#np.linspace(0,2,11)
Ls = [(3,4)] # Important! its Lx  then Ly !!!!

a = 1
Nf = 0
Nrs=[4]
charge_mods = [None]

n = 0
for m in ms:
    for g in gs:
        for Nr in Nrs:
            for Lx, Ly  in Ls:
                for Lstring in range(0, Lx+1, 2):
                    zero_rotor_state= [str(2**(i-1)) if i !=  Nr-1 else str(-2**(i-1)) for i in range(Nr-1, 0, -1)] + ['1'] if Nr != 1 else ['0']
                    one_rotor_state = [str(-2**(i-1)) if i !=  Nr-1 else str(2**(i-1)) for i in range(Nr-1, 0, -1)] + ['0'] if Nr != 1 else ['1']
                    state = [[['empty' if (i_x+i_y)%2==0 else 'full']*Nf+2*zero_rotor_state for i_y in range(Ly)] for i_x in range(Lx)]
                    y_string = Ly//2
                    for i in range(Lx//2 - Lstring//2, Lx//2 + Lstring//2):
                        #print(i)
                        state[i][y_string][Nf:Nf+Nr] = one_rotor_state
                    state
                    inital_state_params = LatticeProductStateConfig(product_state=state)
                    for c_mod in charge_mods:
                        model_params = QED3VRQubitEncodingConfig(Lx=Lx, Ly=Ly, g=g, Nf=Nf, m=[m]*Nf, a=a, Nr=Nr, compress_mpo=True, charge_mod=c_mod)
                        config = DMRGSimulationRampChiConfig(
                            model_class=model_params.model_name,
                            model_params=model_params.model_dump(),
                            initial_state_params=inital_state_params,
                            algorithm_class="TwoSiteDMRGEngine",
                            algorithm_params=dmrg_params,
                            save_psi=False,
                            measure_at_algorithm_checkpoints=True,
                            canonicalize_before_measurement=True,
                        )
                            # Create a signac job with the configuration
                        job = project.open_job(statepoint=config.model_dump())
                        job.init()
                        job.doc['initilized'] = {'date': str(np.datetime64('now')), 'reason':"Compare charge conservation vs non", "Lstring":Lstring}    # Write the configuration to the job's directory
                        #job.remove()
                        n += 1

# Run the simulation and pre analysis:

```{bash}
row submit --action=init_tenpy_simulation_run
row submit --action=run_tenpy_simulation_local -n=4
row scan
row submit --action=fit_entropy_per_unit $(signac find '{"model_class":"MassiveSchwingerModelQubitEncoding"}')
row submit --action=finite_chi_scaling
```